# 00 — Setup and sanity check (local)

Run this first to confirm the local environment and data are wired up correctly before doing real work in another notebook. Assumes: you've cloned this repo, run `uv sync`, and already have `EHRSHOT_ASSETS.zip` downloaded to `~/ehrshot-data/` (see `DATA_ACCESS.md`).

Colab is no longer the primary path — see `scripts/colab_setup.py` and `notebooks/README.md` if you ever need it as a fallback, but everything here assumes local.

In [ ]:
import sys
from pathlib import Path

# scripts/ isn't an installed package (unlike ehrgraph, which uv sync already
# made importable anywhere) -- find the repo root from wherever Jupyter
# actually launched from, so this works whether that's notebooks/, the repo
# root, or VS Code's own default.
_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()

In [ ]:
# Sanity check: confirm the extracted assets look right before doing anything else

assets_dir = data_root / "EHRSHOT_ASSETS"
for sub in ["benchmark", "features", "models", "results", "splits", "data"]:
    p = assets_dir / sub
    print(f"{sub:12s} exists={p.exists()}")

In [ ]:
import pandas as pd

labels = pd.read_csv(assets_dir / "benchmark" / "all_labels.csv")
print(labels.shape)
labels.head()

In [ ]:
import ehrgraph

print("ehrgraph package imports cleanly:", ehrgraph.__name__)

## Verify meds_reader against the real data (issue #1)

Confirmed working locally (Sep 20) -- installs cleanly, no compiled-extension issues on this machine. Run the cell below and confirm it prints real events with no errors.

In [ ]:
from ehrgraph.data.loader import open_database

db_path = data_root / "meds_reader_omop_ehrshot"
db = open_database(str(db_path))

first_subject_id = next(iter(db))
subject = db[first_subject_id]
print(f"subject_id={subject.subject_id}")
for i, event in enumerate(subject.events):
    if i >= 5:
        break
    print(f"  {event.time}  {event.code}")